In [2]:
import fitz
import pandas as pd

pdf = fitz.open("Stock Trader's Almanac 2026_L.pdf")

print("Pages:", len(pdf))

Pages: 273


Enter month

In [3]:
month = "OCTOBER"

Search file keywords

In [4]:
pages_found = []

for page_num in range(len(pdf)):

    text = pdf[page_num].get_text()

    if f"{month} ALMANAC" in text:

        pages_found.append(page_num + 1)

print("Found pages:", pages_found)

Found pages: [3, 127]


Select real page

In [5]:
real_page = max(pages_found)

print("Real Almanac Page:", real_page)

Real Almanac Page: 127


Show the content

In [6]:
page = pdf[real_page - 1]

text = page.get_text()

print(text[:5000])

OCTOBER ALMANAC
Market Probability Chart above is a graphic representation of the S&P 500 Recent
Market Probability Calendar on page 126.
◆ Beware “Octoberphobia” from crashes in 1929, 1987, 554-point drop October 27,
1997, back-to-back massacres 1978 and 1979, Friday the 13th 1989 and the 2008
meltdown ◆ Yet October is a “Bear Killer” and turned the tide in 13 post-WWII bear
markets: 1946, 1957, 1960, 1962, 1966, 1974, 1987, 1990, 1998, 2001, 2002, 2011, and
2022 ◆ First October Dow top in 2007 ◆ Worst six months of the year ends with
October (page 54) ◆ No longer worst month (pages 52 & 60) ◆ Best Dow, S&P and
NASDAQ month from 1993 to 2007 ◆ Midterm election year Octobers since 1950, #1
Dow (+3.2), #1 S&P (+3.0%) and #2 NASDAQ (+3.2%) ◆ October is a great time to
buy ◆ Big October Dow gains five years 1999–2003 after atrocious Septembers ◆
Enter Best Six Months earlier using MACD (page 56) ◆ October 2022, best Dow
month by points, up over 4000 (+14.0%)
October Vital Statistics
DJIA


Preserve evidence

In [7]:
pix = page.get_pixmap(
    matrix=fitz.Matrix(4,4)
)

image_name = f"{month.lower()}_vital_statistics.png"

pix.save(image_name)

print(image_name)

october_vital_statistics.png


In [14]:
import re

blocks = page.get_text("dict")["blocks"]

lines = []

for block in blocks:
    if "lines" not in block:
        continue

    for line in block["lines"]:
        text = "".join(span["text"] for span in line["spans"]).strip()
        if text:
            lines.append(text)

def find_line(keyword):
    for i, line in enumerate(lines):
        if line.strip().lower() == keyword.lower():
            return i
    raise ValueError(f"{keyword} not found")

rank_idx = find_line("Rank")
up_idx = find_line("Up")
down_idx = find_line("Down")
avg_idx = next(i for i, x in enumerate(lines) if x.startswith("Average"))
mid_idx = next(i for i, x in enumerate(lines) if x.startswith("Midterm"))
best_idx = next(i for i, x in enumerate(lines) if x.startswith("Best & Worst"))

def read_values(start, end):

    nums = []

    for line in lines[start+1:end]:
        nums.extend(
            re.findall(r'-?\d+\.?\d*', line.replace("–", "-"))
        )

    return nums

rank_row = read_values(rank_idx, up_idx)
up_row = read_values(up_idx, down_idx)
down_row = read_values(down_idx, avg_idx)
avg_row = read_values(avg_idx, mid_idx)
mid_row = read_values(mid_idx, best_idx)

print(rank_row)
print(up_row)
print(down_row)
print(avg_row)
print(mid_row)

rank = int(rank_row[1])
up_years = int(up_row[1])
down_years = int(down_row[1])
avg_return = float(avg_row[1])
midterm_return = float(mid_row[1])

print()
print("Rank:", rank)
print("Up Years:", up_years)
print("Down Years:", down_years)
print("Average Return:", avg_return)
print("Midterm Return:", midterm_return)

['7', '7', '8', '7', '11']
['44', '44', '29', '28', '26']
['31', '31', '25', '18', '20']
['0.7', '0.9', '0.7', '0.9', '-0.2']
['3.2', '3.0', '3.2', '3.9', '3.2']

Rank: 7
Up Years: 44
Down Years: 31
Average Return: 0.9
Midterm Return: 3.0


Almanac Data Table

In [15]:
stats_df = pd.DataFrame({
    "Metric": [
        "Rank",
        "Up Years",
        "Down Years",
        "Average Return (%)",
        "Midterm Return (%)"
    ],
    "Value": [
        rank,
        up_years,
        down_years,
        avg_return,
        midterm_return
    ]
})

stats_df

,Metric,Value
0,Rank,7.0
1,Up Years,44.0
2,Down Years,31.0
3,Average Return (%),0.9
4,Midterm Return (%),3.0


Verdict is automatically generated based on Almanac data.

In [16]:
if midterm_return >= 1:
    verdict = "Bullish"

elif midterm_return > 0:
    verdict = "Slightly Bullish"

elif midterm_return > -1:
    verdict = "Slightly Bearish"

else:
    verdict = "Bearish"

print("Verdict:", verdict)

Verdict: Bullish


Evidence Table

In [17]:
evidence_df = pd.DataFrame({
    "Evidence": [
        f"{month.title()} Average Return {avg_return}%",
        f"{month.title()} Midterm Return {midterm_return}%"
    ],
    "Impact": [
        "Bullish" if avg_return > 0 else "Bearish",
        "Bullish" if midterm_return > 0 else "Bearish"
    ]
})

evidence_df

,Evidence,Impact
0,October Average Return 0.9%,Bullish
1,October Midterm Return 3.0%,Bullish


Export Evidence CSV file

In [18]:
evidence_df.to_csv(
    f"{month.lower()}_almanac_evidence.csv",
    index=False
)

print(
    f"{month.lower()}_almanac_evidence.csv created"
)

october_almanac_evidence.csv created


Generate Almanac Markdown report

In [19]:
md_text = f"""
# Almanac Agent Output

## Month

{month.title()} 2026

## Statistics

Rank: {rank}

Up Years: {up_years}

Down Years: {down_years}

Average Return: {avg_return}%

Midterm Return: {midterm_return}%

## Final Verdict

{verdict}

## Evidence

- {month.title()} Average Return {avg_return}%
- {month.title()} Midterm Return {midterm_return}%
"""

with open(
    f"{month.lower()}_almanac.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(md_text)

print(
    f"{month.lower()}_almanac.md created"
)

october_almanac.md created
